# Healthy Babies/SfL Analysis

This notebook reads in the input datasets used to run the routing tool for projects relevant to SfL centres, and produces outputs that can be used for further analysis.

* Load LSOA PWC and population data, SfL data, and shapefiles for LSOAs, ICBs and parliamentary constituencies.
* For driving and walking respectively, collect one row per LSOA, showing how many SfLs are within 5, 10, 15, etc. minutes of each LSOA's PWC, as well as details about the closest SfL (time-wise).
* For driving and walking respectively, produce population-weighted averages of ICB and parliamentary constituency summary statistics - using lookups of LSOAs to ICBs and LSOAs to constituencies.
* Plot basic maps of some summary statistics, to act as demonstrations.
* Save datasets to the `output` folder for exporting to Sharepoint.

In [0]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from matplotlib import colors

from src.datavis import plot_map
from src.read_write_data import load_config, read_in_route, write_dataset
from src.stats import (
    aggregate_stats_by_geography,
    aggregate_summary_upto_larger_geography,
    bin_values,
    summarise_stats
)

config: dict = load_config("../../config.yml")

## Read in data

In [0]:
lsoa_pwcs: pd.DataFrame = (
    pd.read_csv(config["lsoa_pwc_clean_data_file_path"])
    .merge(
        pd.read_csv(config["lsoa_rural_urban_classification"]),
        how="left",
        left_on="unique_code",
        right_on="LSOA21CD",
        validate="one_to_one",
    )
    .merge(
        pd.read_csv(config["lsoa_population_data"]),
        how="left",
        left_on="unique_code",
        right_on="LSOA 2021 Code"
    )
    .rename(columns={"Total": "population"})
    .assign(population=lambda df: df.population.str.replace(',', '').astype(int))
)[
    ["unique_code", "lat", "lon", "LSOA21NM", "RUC21CD", "RUC21NM", "Urban_rural_flag", "population"]
]
lsoa_pwcs.head(1)

In [0]:
sfls: pd.DataFrame = (
    pd.read_csv(config["sfl_data_file_path"],
                     dtype={"unique_code": 'string'},
                     encoding="latin-1")
)
sfls.head(1)

sfl_LAs_unique = sfls[["id", "LA NAME"]].drop_duplicates()
print(sfl_LAs_unique)
print(len(sfl_LAs_unique))

In [0]:
lsoa_to_icb_df: pd.DataFrame = pd.read_csv(config["lsoa_to_icb"])
lsoa_to_icb_df.head(1)

In [0]:
icb_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/ICB_APR_2023_EN_BSC.shp"
)
icb_boundaries_en.head(1)

In [0]:
lsoa_to_constituency_df: pd.DataFrame = gpd.read_file(
    "../../input/map/LSOA_(2021)_to_Westminster_Parliamentary_Constituencies_(July_2024)_Best_Fit_Lookup_in_EW.shp"
)[["LSOA21CD", "PCON24CD", "PCON24NM"]]
lsoa_to_constituency_df.head(5)

In [0]:
constituency_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/PCON_JULY_2024_UK_BFE.shp"
).query("PCON24CD.str[0] == 'E'")
constituency_boundaries_en.head(1)

In [0]:
lsoa_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/LSOA_2021_EW_BSC_V4.shp"
).query("LSOA21CD.str[0] == 'E'")
lsoa_boundaries_en.head(1)

In [0]:
# Requires 
# Lower_Layer_Super_Output_Area_(2021)to_LAD(April_2023)_Lookup_in_England_and_Wales 
# https://geoportal.statistics.gov.uk/datasets/9dd1fe173d894b6a838b5ee016d3037e_0/about
lsoa_to_utla_df: pd.DataFrame = (
    pd.read_csv(
        "../../input/Lower_Layer_Super_Output_Area_(2021)_to_LAD_(April_2023)_Lookup_in_England_and_Wales.csv"
    )
    .query("LAD23CD.str[0] == 'E'")
    .merge(
        pd.read_csv(
            "../../input/Local_Authority_District_to_County_and_Unitary_Authority_(April_2023)_Lookup_in_EW.csv"
        ),
        left_on="LAD23CD",
        right_on="LTLA23CD",
        how="left",
    )[["LSOA21CD", "LTLA23CD", "UTLA23CD", "LSOA21NM", "LTLA23NM", "UTLA23NM"]]
)
print(lsoa_to_utla_df)

In [0]:
ltla_boundaries_en = (
    gpd.read_file(
        "../../input/map/LAD_MAY_2023_UK_BFE_V2.shp"
    )
    .query("LAD23CD.str[0] == 'E'")
    .merge(
        lsoa_to_utla_df.drop_duplicates("LTLA23CD")[["LTLA23CD", "UTLA23CD"]],
        how="left",
        left_on="LAD23CD",
        right_on="LTLA23CD"
    )
)
ltla_boundaries_en.head(2)

In [0]:
utla_boundaries_en: gpd.GeoDataFrame = (
    ltla_boundaries_en
    .dissolve("UTLA23CD")
    .reset_index()
    .drop(columns=["LAD23CD", "LAD23NM", "LAD23NMW"])
)
utla_boundaries_en.head(2)

In [0]:
# Read in car access by LSOA data (download from ONS dataset Dataset ID: TS045, making sure you change the area type to LSOA)
# Rename so it has the column names:  
# LSOA_code, car_availability_code, number
car_access_df = (
    pd.read_csv("../../input/LSOA_data/Car_access_by_LSOA.csv", header=0)
    .query("car_availability_code != -8")
    .assign(
        car_availability_code=lambda df: df.car_availability_code.astype(int),
        number=lambda df: df.number.astype(int)
    )
)

print(car_access_df)

In [0]:
# Read in CLIF by LSOA (from Stat-Xplore website, based on relative low income [as opposed to absolute low income]). Extra rows/columns may need to be removed and add the following column headers: LSOA and CLIF

# For the age 0-2 CLIF, you will need to take the Stat-Xplore table with GB -> LSOA as rows and add columns for ages 0-2. When you download the table, it will automatically create a total column. Delete the columns for each age and rename the remaining columns "LSOA" and "CLIF". Delete the rows at the bottom
if config["all_age_clif"]:
     clif_df = pd.read_csv("../../input/LSOA_data/CLIF_202324.csv", header=0)
else: 
    clif_df = pd.read_csv("../../input/LSOA_data/CLIF_age_0_to_2_2023_24.csv", header=0)

# Convert CLIF to float
clif_df['CLIF'] = pd.to_numeric(clif_df['CLIF'], errors='coerce')
# Replace NaN with 0
clif_df['CLIF'] =clif_df['CLIF'].fillna(0)

print(clif_df)

## Cars

In [0]:
lsoa_sfls_car = read_in_route(
    config["database_path"], "lsoa_pwcs", "start_for_life", "car", 20
)
lsoa_sfls_car_stats: pd.DataFrame = summarise_stats(
    lsoa_sfls_car, lsoa_pwcs, sfls, [5, 10, 15, 20, 30]
)
lsoa_sfls_car_stats

In [0]:
plot_map(
    lsoa_sfls_car_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_15_minutes",
    "LSOA21CD",
    higher_is_worse=False,
    #norm=colors.TwoSlopeNorm(vcenter=30, vmin=0, vmax=50),
)
plot_map(
    lsoa_sfls_car_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
    #norm=colors.TwoSlopeNorm(vcenter=10, vmin=0, vmax=30),
)

In [0]:
plot_map(
    lsoa_sfls_car_stats,
    lsoa_to_utla_df,
    utla_boundaries_en,
    "number_of_locations_within_15_minutes",
    "UTLA23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_car_stats,
    lsoa_to_utla_df,
    utla_boundaries_en,
    "min_time",
    "UTLA23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_sfls_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_15_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_sfls_car_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_15_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_car_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
    norm=colors.TwoSlopeNorm(vcenter=5, vmin=1, vmax=9)
)

In [0]:
(
    bin_values(lsoa_sfls_car_stats, "min_time", [0, 5, 10, 15, 20, 25, 30])
    .groupby(["min_time_binned"])
    .agg({"population": sum})
    .reset_index()
    .query("population > 0")
)

In [0]:
(
    bin_values(lsoa_sfls_car, "time", [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150])
    .groupby(["uniqueCodeL", "time_binned"])
    .agg({"uniqueCodeR": len})
    .reset_index()
    .query("~uniqueCodeR.isnull()")
)

## Walking

In [0]:
lsoa_sfls_foot = read_in_route(
    config["database_path"], "lsoa_pwcs", "start_for_life", "foot", n = 20
)
lsoa_sfls_foot_stats: pd.DataFrame = summarise_stats(
    lsoa_sfls_foot, lsoa_pwcs, sfls, [5, 10, 15, 20, 30]
)
lsoa_sfls_foot_stats

In [0]:
aggregate_summary_upto_larger_geography(
    lsoa_sfls_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
)

In [0]:
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_30_minutes",
    "LSOA21CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
    norm=colors.TwoSlopeNorm(vcenter=10, vmin=0, vmax=30),
)

In [0]:
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_to_utla_df,
    utla_boundaries_en,
    "number_of_locations_within_30_minutes",
    "UTLA23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_to_utla_df,
    utla_boundaries_en,
    "min_time",
    "UTLA23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_30_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_sfls_foot_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
    # norm=colors.TwoSlopeNorm(vcenter=5, vmin=1, vmax=9)
)

In [0]:
(
    bin_values(
        lsoa_sfls_foot_stats,
        "min_time",
        [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360, 420])
    .groupby(["min_time_binned"])
    .agg({"population": sum})
    .reset_index()
    .query("population > 0")
)

In [0]:
lsoa_sfls_car[["time", "distance"]].equals(lsoa_sfls_foot[["time", "distance"]])


In [0]:
(
    bin_values(lsoa_sfls_foot, "time", [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360, 420, 480, 540, 600, 2400])
    .groupby(["uniqueCodeL", "time_binned"])
    .agg({"uniqueCodeR": len})
    .reset_index()
    .query("~uniqueCodeR.isnull()")
)

In [0]:
# Find number of LSOAs within 30 minutes walk of a SfL centre


## Demographic analysis

In [0]:
# Find the number of people who have access to an SfL hub within 30 minutes walk

pop_within_30_mins_walk = lsoa_sfls_foot_stats[lsoa_sfls_foot_stats["number_of_locations_within_30_minutes"] > 0]["population"].sum()
total_pop = lsoa_sfls_foot_stats["population"].sum()

perc_pop_within_30_mins_walk = round(100 * pop_within_30_mins_walk / total_pop, 2)

print(pop_within_30_mins_walk)
print(total_pop)
print(perc_pop_within_30_mins_walk)

In [0]:
# Group car access by LSOA code and find percentage with car access
car_access_grouped = (
    car_access_df.groupby("LSOA_code")
      .apply(lambda x: x.loc[x["car_availability_code"] != 0, "number"].sum() 
                       / x["number"].sum())
      .reset_index(name="prop_car_access")
      .rename(columns={'LSOA_code': 'lsoa_id'})
)

display(car_access_grouped)

In [0]:
# Join car availability to LSOA data for walking and driving (30 minute only)
# dfs with just 30 minutes walking/driving
LSOA_30_walking = lsoa_sfls_foot_stats[["lsoa_id", "LSOA21NM", "population", "number_of_locations_within_30_minutes"]]
LSOA_30_driving = lsoa_sfls_car_stats[["lsoa_id", "number_of_locations_within_30_minutes"]]

# Merge the dfs
LSOA_driving_walking = car_access_grouped.merge(LSOA_30_walking, on = "lsoa_id").merge(LSOA_30_driving, on = "lsoa_id", suffixes = ("_walking", "_driving"))

# Merge with CLIF data at LSOA level
LSOA_driving_walking_CLIF = LSOA_driving_walking.merge(clif_df, left_on = "LSOA21NM", right_on ="LSOA", how = "left")

# Find areas where the CLIF data hasn't matched to an LSOA
LSOA_driving_walking_CLIF_missing = LSOA_driving_walking_CLIF[LSOA_driving_walking_CLIF["LSOA"].isna()]
display(LSOA_driving_walking_CLIF_missing)
print(len(LSOA_driving_walking_CLIF_missing))



In [0]:
#Calculate the proportion of people within 30 minutes (based on car access)
LSOA_driving_walking_CLIF["30_min_access"] = (
    LSOA_driving_walking_CLIF["prop_car_access"] * (LSOA_driving_walking_CLIF["number_of_locations_within_30_minutes_driving"] > 0)
    + (1 - LSOA_driving_walking_CLIF["prop_car_access"]) * (LSOA_driving_walking_CLIF["number_of_locations_within_30_minutes_walking"] > 0)
)

pd.set_option("display.max_columns", None)
display(LSOA_driving_walking_CLIF)

In [0]:
# Find areas with no walking access
LSOA_driving_walking_filter = LSOA_driving_walking_CLIF[LSOA_driving_walking_CLIF["number_of_locations_within_30_minutes_walking"] == 0]
display(LSOA_driving_walking_filter)

In [0]:
# find the UTLAs where there are SfLs
# Create LTLA to UTLA lookup
ltla_to_utla_lookup = lsoa_to_utla_df[["LTLA23CD", "UTLA23CD", "UTLA23NM"]].drop_duplicates()

# Join the SfL list to the LTLA to UTLA lookup 
sfl_LAs_unique_merge = sfl_LAs_unique.merge(ltla_to_utla_lookup, 
                                        how = "left",
                                        left_on = "id",
                                        right_on = "LTLA23CD")

# Some SfLs are already listed with the UTLA name, 
# but some with the LTLA code/name. 
# Where they are LTLA and therefore the matched UTLA is present,
# replace id with UTLA
sfl_LAs_unique_merge["id"] = ( 
    sfl_LAs_unique_merge["id"]
        .where(sfl_LAs_unique_merge["UTLA23CD"].isna(),
                sfl_LAs_unique_merge["UTLA23CD"],
        )
)

sfl_LAs_unique_merge["LA NAME"] = ( 
    sfl_LAs_unique_merge["LA NAME"]
        .where(sfl_LAs_unique_merge["UTLA23NM"].isna(),
                sfl_LAs_unique_merge["UTLA23NM"],
        )
)
display(sfl_LAs_unique_merge)


In [0]:
# Show the distinct UTLAs with SfLs in them
distinct_SFL_LAs = sfl_LAs_unique_merge[["LA NAME"]].drop_duplicates()
display(distinct_SFL_LAs)
print(len(distinct_SFL_LAs))

In [0]:
# Restrict to LAs where there are SfLs
if not config["all_LAs"]:
    # Inner join so that we only keep LSOAs in a UTLA with an SfL centre
    LSOA_driving_walking_CLIF_matched = (
        LSOA_driving_walking_CLIF
            .merge(lsoa_to_utla_df, 
                  how = "inner",
                  left_on = "lsoa_id",
                  right_on = "LSOA21CD",)
            .merge(sfl_LAs_unique_merge,
                   how = "inner",
                   left_on = "UTLA23CD",
                   right_on = "id",)
    )
else: 
    # Else just use the existing dataframe
    LSOA_driving_walking_CLIF_matched = LSOA_driving_walking_CLIF

# Find distinct LAs in the new data to check they are all there
distinct_LAs = LSOA_driving_walking_CLIF_matched[["id"]].drop_duplicates()
display(distinct_LAs)
print(len(distinct_LAs))





In [0]:
display(LSOA_driving_walking_CLIF_matched)

In [0]:

# Find the number of people who have access to an SfL hub within 30 minutes (weighted by car access)
pop_within_30_mins_weighted = (LSOA_driving_walking_CLIF_matched["30_min_access"] * LSOA_driving_walking_CLIF_matched["population"]).sum()
total_pop = LSOA_driving_walking_CLIF_matched["population"].sum()

perc_pop_within_30_mins_weighted = round(100 * pop_within_30_mins_weighted / total_pop, 2)

print(pop_within_30_mins_weighted)
print(total_pop)
print(perc_pop_within_30_mins_weighted)


In [0]:

# Find the number of CLIF who have access to an SfL hub within 30 minutes (weighted by car access)
CLIF_within_30_mins_weighted = (LSOA_driving_walking_CLIF_matched["30_min_access"] * LSOA_driving_walking_CLIF_matched["CLIF"]).sum()
total_CLIF = LSOA_driving_walking_CLIF_matched["CLIF"].sum()

perc_CLIF_within_30_mins_weighted = round(100 * CLIF_within_30_mins_weighted / total_CLIF, 2)

print(CLIF_within_30_mins_weighted)
print(total_CLIF)
print(perc_CLIF_within_30_mins_weighted)

In [0]:
# Change 30 mins walking/driving columns to be either 1 or 0
LSOA_driving_walking_CLIF_binary = LSOA_driving_walking_CLIF_matched.copy()
LSOA_driving_walking_CLIF_binary['30_mins_walk'] = (LSOA_driving_walking_CLIF_binary['number_of_locations_within_30_minutes_walking'] > 0).astype(int)
LSOA_driving_walking_CLIF_binary["30_mins_drive"] = (LSOA_driving_walking_CLIF_binary["number_of_locations_within_30_minutes_driving"] != 0).astype(int)

# Find the number of people within 30 minutes walk, drive and weighted access
cols = ["30_mins_walk", "30_mins_drive", "30_min_access"]
access_national = pd.DataFrame([{
    **{col: (LSOA_driving_walking_CLIF_binary[col] * LSOA_driving_walking_CLIF_binary["population"]).sum()
    for col in cols},
    "population": LSOA_driving_walking_CLIF_binary["population"].sum()
}])

print(access_national)


In [0]:
cols = ["30_mins_walk", "30_mins_drive", "30_min_access"]

# Find the proportion of people nationally within 30 minutes walk, drive and weighted access
access_prop = access_national[cols].div(access_national["population"], axis=0)
print(access_prop)


In [0]:
# Find the number of CLIF within 30 minutes walk, drive and weighted access
cols = ["30_mins_walk", "30_mins_drive", "30_min_access"]
CLIF_access_national = pd.DataFrame([{
    **{col: (LSOA_driving_walking_CLIF_binary[col] * LSOA_driving_walking_CLIF_binary["CLIF"]).sum()
    for col in cols},
    "CLIF": LSOA_driving_walking_CLIF_binary["CLIF"].sum()
}])

print(CLIF_access_national)

In [0]:
cols = ["30_mins_walk", "30_mins_drive", "30_min_access"]

# Find the proportion of CLIF nationally within 30 minutes walk, drive and weighted access
CLIF_access_prop = CLIF_access_national[cols].div(CLIF_access_national["CLIF"], axis=0)
print(CLIF_access_prop)

In [0]:
# Aggregate data up to UTLA (population-weighted)

lsoa_utla_simp = lsoa_to_utla_df[["LSOA21CD", "UTLA23NM"]]
UTLA_driving_walking = LSOA_driving_walking_CLIF_binary.merge(lsoa_utla_simp, left_on = "lsoa_id", right_on = "LSOA21CD")
print(UTLA_driving_walking)

cols = ["30_mins_walk", "30_mins_drive", "30_min_access", "prop_car_access"]

UTLA_access = (
    UTLA_driving_walking.groupby("UTLA23NM")
      .apply(lambda x: pd.Series({
          **{col: (x[col] * x["population"]).sum() / x["population"].sum()
          for col in cols},
          "population": x["population"].sum()
      }))
      .reset_index()
      .rename(columns={'UTLA23NM': 'UTLA'})
)
print(UTLA_access)

## Output data

In [0]:
lsoa_sfls_car_stats_output = (
    lsoa_sfls_car_stats
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "min_time_location_id",
        "lat_lon_location",
    ]]
)
lsoa_sfls_car_stats_output.head(1)

In [0]:
lsoa_sfls_car_stats_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_sfls_car_stats_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
lsoa_sfls_car_stats_ICB_output.head(1)

In [0]:
lsoa_sfls_car_stats_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_sfls_car_stats_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
lsoa_sfls_car_stats_PCON_output.head(1)

In [0]:
lsoa_sfls_foot_stats_output = (
    lsoa_sfls_foot_stats
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "min_time_location_id",
        "lat_lon_location",
    ]]
)
lsoa_sfls_foot_stats_output.head(1)

In [0]:
lsoa_sfls_foot_stats_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_sfls_foot_stats_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
lsoa_sfls_foot_stats_ICB_output.head(1)

In [0]:
lsoa_sfls_foot_stats_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_sfls_foot_stats_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
lsoa_sfls_foot_stats_PCON_output.head(1)

In [0]:
write_dataset(
    lsoa_sfls_car
    .merge(
        lsoa_to_icb_df[["LSOA21CD", "ICB23CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD")
    .merge(
        lsoa_to_constituency_df[["LSOA21CD", "PCON24CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD"),
    "../../output/full_output_LSOAs_to_SFLSs_car_with_ICB_PCON.parquet"
)

In [0]:
write_dataset(
    lsoa_sfls_foot
    .merge(
        lsoa_to_icb_df[["LSOA21CD", "ICB23CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD")
    .merge(
        lsoa_to_constituency_df[["LSOA21CD", "PCON24CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD"),
    "../../output/full_output_LSOAs_to_SFLs_foot_with_ICB_PCON.parquet"
)

In [0]:
write_dataset(
    lsoa_sfls_car_stats_output,
    "../../output/summary_data_LSOAs_to_SFLs_car.xlsx"
)

In [0]:
write_dataset(
    lsoa_sfls_foot_stats_output,
    "../../output/summary_data_LSOAs_to_SFLs_foot.xlsx"
)

In [0]:
write_dataset(
    lsoa_sfls_car_stats_ICB_output,
    "../../output/summary_data_ICBs_to_SFLs_car.xlsx"
)

In [0]:
write_dataset(
    lsoa_sfls_foot_stats_ICB_output,
    "../../output/summary_data_ICBs_to_SFLs_foot.xlsx"
)

In [0]:
write_dataset(
    lsoa_sfls_car_stats_PCON_output,
    "../../output/summary_data_PCONs_to_SFLs_car.xlsx"
)

In [0]:
write_dataset(
    lsoa_sfls_car_stats_PCON_output,
    "../../output/summary_data_PCONs_to_SFLs_foot.xlsx"
)